In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# E类型，19
# input: 3*128*128
class VGGNet(nn.Module):
    def __init__(self, num_classes=101):
        super(VGGNet, self).__init__()

        self.features = nn.Sequential(
          #1
          nn.Conv2d(3, 64, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(64, 64, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2, stride=2),
          #2
          nn.Conv2d(64, 128, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(128, 128, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2, stride=2),
          #3
          nn.Conv2d(128, 256, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(256, 256, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(256, 256, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(256, 256, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2, stride=2),
          #4
          nn.Conv2d(256, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2, stride=2),
          #5
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.Conv2d(512, 512, kernel_size=3, padding=1),
          nn.ReLU(inplace=True),
          nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
          nn.Linear(512 * 4 * 4, 4096),
          nn.ReLU(inplace=True),
          nn.Dropout(0.5),
          nn.Linear(4096, 4096),
          nn.ReLU(inplace=True),
          nn.Dropout(0.5),
          nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
    
    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
      

In [24]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import matplotlib.pyplot as plt


class DataSetLoader_test(Dataset):
    def __init__(self, root_dir, transform=None, num_classes=101, return_pil=False):
        self.root_dir = root_dir
        self.transform = transform
        self.num_classes = num_classes
        # 如果 return_pil 为 True，则 __getitem__ 返回 PIL 图像而不是张量
        self.return_pil = return_pil

        # 收集所有 (img_path, label_idx)
        self.samples = []
        self.class_names = sorted(
            [
                d
                for d in os.listdir(root_dir)
                if os.path.isdir(os.path.join(root_dir, d))
            ]
        )[: self.num_classes]
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_dir = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    img_path = os.path.join(cls_dir, fname)
                    label = self.class_to_idx[cls_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.return_pil:
            return image, label

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [25]:
import random

class DataSetLoader_train(Dataset):
  def __init__(self, root_dir, transform=None, num_classes=101, split_ratio=0.8, return_pil=False, dataclass="train"):
    self.root_dir = root_dir
    self.transform = transform
    self.num_classes = num_classes
    self.split_ratio = split_ratio
    self.return_pil = return_pil
    self.dataclass = dataclass
    self.trainSamples = []
    self.validSamples = []
    self.class_names = sorted(
      [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    )[: self.num_classes]
    self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.class_names)}
    self.train_count = {cls_name: 0 for cls_name in self.class_names}
    self.valid_count = {cls_name: 0 for cls_name in self.class_names}

    for cls_name in self.class_names:
      cls_dir = os.path.join(root_dir, cls_name)
      cls_samples = []
      for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
          img_path = os.path.join(cls_dir, fname)
          label = self.class_to_idx[cls_name]
          cls_samples.append((img_path, label))
      random.shuffle(cls_samples)
      split_idx = int(len(cls_samples) * self.split_ratio)
      train_split = cls_samples[:split_idx]
      valid_split = cls_samples[split_idx:]
      self.trainSamples.extend(train_split)  # 前80% 训练样本
      self.validSamples.extend(valid_split)  # 后20% 验证样本
      self.train_count[cls_name] += len(train_split)
      self.valid_count[cls_name] += len(valid_split)

  def __len__(self):
    if self.dataclass == "train":
      return len(self.trainSamples)
    else:
      return len(self.validSamples)

  def __getitem__(self, idx):
    if self.dataclass == "train":
      img_path, label = self.trainSamples[idx]
    else:
      img_path, label = self.validSamples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.return_pil:
      return image, label

    if self.transform is not None:
      image = self.transform(image)
    else:
      transform = T.Compose(
        [
          T.Resize((128, 128)),
          T.ToTensor(),
          T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
      )
      image = transform(image)

    return image, label

In [26]:
from tqdm import tqdm


def test_val(model, val_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="验证中"):
            images, labels = images.to(device), labels.to(device)
            if images.dim() == 5:
                bsz, ncrops, c, h, w = images.size()
                outputs = model(images.view(-1, c, h, w))
                outputs = outputs.view(bsz, ncrops, -1).mean(dim=1)
            else:
                outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / max(total, 1)
    accuracy = correct / max(total, 1)
    return accuracy

In [27]:
def train(
    model,
    epoch,
    train_loader,
    optimizer,
    criterion,
    device,
    scheduler=None,
    val_loader=None,
):
    train_acc = []
    val_acc = []
    loss_history = []

    for ep in range(epoch):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"训练中 [{ep+1}/{epoch}]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            batch_loss = criterion(outputs, labels)
            batch_loss.backward()
            optimizer.step()

            total_loss += batch_loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        avg_loss = total_loss / max(total, 1)
        accuracy = correct / max(total, 1)
        train_acc.append(accuracy)
        loss_history.append(avg_loss)

        val_acc_ = None
        if val_loader is not None:
            val_acc_ = test_val(model, val_loader, criterion, device)
            val_acc.append(val_acc_)

        if scheduler is not None:
            if val_acc_ is None:
                scheduler.step()
            else:
                scheduler.step(val_acc_)

        if val_acc_ is None:
            print(
                f"Epoch [{ep+1}/{epoch}] 平均损失: {avg_loss:.4f} 训练准确率: {accuracy:.4f}"
            )
        else:
            print(
                f"Epoch [{ep+1}/{epoch}] 平均损失: {avg_loss:.4f} 训练准确率: {accuracy:.4f} 验证准确率: {val_acc_:.4f}"
            )

    return train_acc, loss_history, val_acc

In [28]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def _pick_cjk_font():
    candidates = [
        "Noto Sans CJK SC",
        "Noto Sans CJK",
        "SimHei",
        "Microsoft YaHei",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            return name
    return None

def _set_plot_font():
    font_name = _pick_cjk_font()
    if font_name:
        plt.rcParams["font.sans-serif"] = [font_name]
        plt.rcParams["axes.unicode_minus"] = False
        return True
    return False

def plot_metrics(train_acc, val_acc, loss):
    has_cjk_font = _set_plot_font()
    epochs = list(range(1, len(train_acc) + 1))

    plt.figure(figsize=(12, 5))

    ax1 = plt.subplot(1, 2, 1)
    ax1.plot(epochs, train_acc, label='training accuracy')
    if val_acc:
        ax1.plot(epochs, val_acc, label='validation accuracy')
    ax1.set_title('Training and Validation Accuracy' if has_cjk_font else 'Train/Val Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy' if has_cjk_font else 'Accuracy')
    ax1.legend()

    best_train_idx = max(range(len(train_acc)), key=lambda i: train_acc[i])
    ax1.scatter(epochs[best_train_idx], train_acc[best_train_idx], color='C0')
    ax1.annotate(
        f"train={train_acc[best_train_idx]:.3f}",
        (epochs[best_train_idx], train_acc[best_train_idx]),
        textcoords="offset points",
        xytext=(6, 6),
    )
    if val_acc:
        best_val_idx = max(range(len(val_acc)), key=lambda i: val_acc[i])
        ax1.scatter(epochs[best_val_idx], val_acc[best_val_idx], color='C1')
        ax1.annotate(
            f"val={val_acc[best_val_idx]:.3f}",
            (epochs[best_val_idx], val_acc[best_val_idx]),
            textcoords="offset points",
            xytext=(6, 6),
        )

    ax2 = plt.subplot(1, 2, 2)
    ax2.plot(epochs, loss, label='training loss')
    ax2.set_title('Training Loss' if has_cjk_font else 'Train Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss' if has_cjk_font else 'Loss')
    ax2.legend()

    best_loss_idx = min(range(len(loss)), key=lambda i: loss[i])
    ax2.scatter(epochs[best_loss_idx], loss[best_loss_idx], color='C0')
    ax2.annotate(
        f"min={loss[best_loss_idx]:.3f}",
        (epochs[best_loss_idx], loss[best_loss_idx]),
        textcoords="offset points",
        xytext=(6, 6),
    )

    plt.tight_layout()
    plt.show()


In [42]:
import random
from torchvision.transforms.v2 import ScaleJitter

train_dir = "/data/data_taohy/awesomeCopression/food101_images/train"
test_dir = "/data/data_taohy/awesomeCopression/food101_images/validation"


device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
# model = VGGNet(num_classes=101)
# model.init_weights()
# if torch.cuda.device_count() > 1:
#     print("使用多卡训练, GPU 数量:", torch.cuda.device_count())
#     model = nn.DataParallel(model)
# model = model.to(device)


# S = [150,300]
transform_train = T.Compose(
    [
        ScaleJitter((128, 128), (150 / 128, 300 / 128)),
        T.Resize((128, 128)),  # 强制统一尺寸，避免 batch 内大小不一致
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# S = 150
transform_val = T.Compose(
    [
        T.Resize((150, 150)),
        T.CenterCrop((128, 128)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = DataSetLoader_train(
    root_dir=train_dir,
    num_classes=101,
    transform=transform_train,
    split_ratio=0.8,
    return_pil=False,
    dataclass="train",
)
val_dataset = DataSetLoader_train(
    root_dir=train_dir,
    num_classes=101,
    transform=transform_val,
    split_ratio=0.8,
    return_pil=False,
    dataclass="valid",
)


# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8)
# val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=8)

# optimizer = torch.optim.SGD(
#     model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4
# )

# scheduler_base = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer,
#     mode="max",  # 监控准确率（越大越好）
#     factor=0.1,  # 乘以 0.1（即除以 10）
#     patience=5,  # 等待 5 个 epoch 不改善才调整
# )

# criterion = nn.CrossEntropyLoss()

# train_acc, loss, val_acc = train(
#     model,
#     epoch=50,
#     train_loader=train_loader,
#     optimizer=optimizer,
#     criterion=criterion,
#     device=device,
#     scheduler=scheduler_base,
#     val_loader=val_loader,
#  )

In [30]:
import os
from datetime import datetime

# 保存/加载检查点的实用函数（在新的 notebook cell 中执行）
def save_checkpoint(path, model, optimizer, scheduler=None, epoch=None, best_acc=None):
  os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
  # 如果使用了 DataParallel，保存 module 的 state_dict
  state_dict = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
  checkpoint = {
    "epoch": epoch,
    "model_state_dict": state_dict,
    "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
    "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
    "best_acc": best_acc,
    "saved_at": datetime.utcnow().isoformat(),
  }
  torch.save(checkpoint, path)
  print(f"Saved checkpoint to {path}")

In [31]:
# save_checkpoint("vgg19_food101_50——1.pth", model, optimizer, scheduler_base, epoch=50, best_acc=max(val_acc) if val_acc else None)

In [32]:
# plot_metrics(train_acc, val_acc, loss)

In [33]:
# train_acc1, loss1, val_ac1c = train(
#     model,
#     epoch=30,
#     train_loader=train_loader,
#     optimizer=optimizer,
#     criterion=criterion,
#     device=device,
#     scheduler=scheduler_base,
#     val_loader=val_loader,
#  )

In [34]:
# plot_metrics(train_acc + train_acc1, val_acc + val_ac1c, loss + loss1)

In [35]:
# save_checkpoint("vggnet19_food101_80——1.pth", model, optimizer, scheduler_base, epoch=100, best_acc=max(val_acc + val_ac1c))

In [36]:
test_base_transform = T.Compose([T.ToTensor(), T.Resize((128, 128))])


def test_collate_fn(batch):
    images, labels = zip(*batch)
    return list(images), torch.tensor(labels, dtype=torch.long)


test_dataset = DataSetLoader_test(
    root_dir=test_dir,
    num_classes=101,
    transform=test_base_transform,
    return_pil=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    collate_fn=test_collate_fn,
)

In [37]:
def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location=device):
  checkpoint = torch.load(path, map_location=map_location)
  # 如果当前 model 是 DataParallel，需要把 state_dict 加载到 module
  target = model.module if hasattr(model, "module") else model
  target.load_state_dict(checkpoint["model_state_dict"])
  if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
  if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
  print(f"Loaded checkpoint from {path}, epoch={checkpoint.get('epoch')}, best_acc={checkpoint.get('best_acc')}")
  return checkpoint

In [38]:
model = VGGNet(num_classes=101)
load_checkpoint("vggnet19_food101_80.pth", model, optimizer=None, scheduler=None)

if torch.cuda.device_count() > 1:
    print("使用多卡训练, GPU 数量:", torch.cuda.device_count())
    model = nn.DataParallel(model)
model = model.to(device)

Loaded checkpoint from vggnet19_food101_80.pth, epoch=100, best_acc=0.7638943894389439
使用多卡训练, GPU 数量: 4


In [39]:
# 密集评估 (Dense Testing): 将 VGG 的 3 个全连接层映射为卷积层
class DenseVGGNet(nn.Module):
    def __init__(self, trained_vgg):
        super().__init__()
        self.features = trained_vgg.features  # 复用已训练特征层

        fc1 = trained_vgg.classifier[0]  # Linear(512*4*4 -> 4096)
        fc2 = trained_vgg.classifier[3]  # Linear(4096 -> 4096)
        fc3 = trained_vgg.classifier[6]  # Linear(4096 -> num_classes)

        self.classifier_conv = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=4, stride=1, padding=0),  # 对应 fc1 (4x4)
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(4096, 4096, kernel_size=1, stride=1, padding=0),  # 对应 fc2
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(
                4096, fc3.out_features, kernel_size=1, stride=1, padding=0
            ),  # 对应 fc3
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))  # 全局池化到 num_classes x 1 x 1
        self.load_fc_weights(fc1, fc2, fc3)

    def load_fc_weights(self, fc1, fc2, fc3):
        with torch.no_grad():
            self.classifier_conv[0].weight.copy_(
                fc1.weight.view(4096, 512, 4, 4)
            )  # type: ignore # 对应 fc1
            self.classifier_conv[0].bias.copy_(fc1.bias)  # type: ignore

            self.classifier_conv[3].weight.copy_(fc2.weight.view(4096, 4096, 1, 1))  # type: ignore
            self.classifier_conv[3].bias.copy_(fc2.bias)  # type: ignore

            self.classifier_conv[6].weight.copy_(
                fc3.weight.view(fc3.out_features, 4096, 1, 1)
            )  # type: ignore
            self.classifier_conv[6].bias.copy_(fc3.bias)  # type: ignore

    def forward(self, x):
        x = self.features(x)
        x = self.classifier_conv(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)  # [B, num_classes]
        return x


# 从已训练 model 构建 dense model（兼容 DataParallel）
base_model = model.module if hasattr(model, "module") else model
dense_model = DenseVGGNet(base_model).to(device)
if hasattr(model, "module"):
    dense_model = nn.DataParallel(dense_model)

print("Dense model ready.")

Dense model ready.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import vgg19
from PIL import Image
from tqdm import tqdm

class VGG19DenseEvaluator:
    def __init__(self, model, scales=[384, 512, 640], num_classes=1000, device='cuda'):
        self.model = model.to(device)
        self.model.eval()
        self.scales = scales
        self.num_classes = num_classes
        self.device = device
        
        # ImageNet 归一化
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
    
    def preprocess(self, img_pil, scale):
        # 调整图像尺寸
        img_pil = T.Resize((scale, scale))(img_pil)
        
        # 转换为 tensor
        img_tensor = T.ToTensor()(img_pil).to(self.device)
        
        # 归一化
        img_tensor = (img_tensor - self.mean) / self.std
        
        return img_tensor.unsqueeze(0)
    
    @torch.no_grad()
    def predict_single_scale(self, img_pil, scale, flip=True):
        # 原始图像
        img = self.preprocess(img_pil, scale)  # [1, 3, S, S]
        
        # 全卷积前向: [1, 3, S, S] -> [1, num_classes, h, w]
        out = self.model(img)
        
        # 空间平均池化: 对空间维度取平均 [1, C, h, w] -> [1, C]
        out = out.mean(dim=[2, 3])
        
        # 水平翻转增强 (论文标准做法)
        if flip:
            img_flip = torch.flip(img, dims=[3])  # 水平翻转
            out_flip = self.model(img_flip)
            out_flip = out_flip.mean(dim=[2, 3])
            out = (out + out_flip) / 2.0
        
        return out.squeeze()  # [num_classes]
    
    @torch.no_grad()
    def predict_multiscale(self, img_pil, flip=True):
        """
        多尺度密集预测 (论文标准方法)
        
        对每个尺度进行密集评估，然后平均所有尺度的结果
        """
        outputs = []
        
        for scale in self.scales:
            out = self.predict_single_scale(img_pil, scale, flip=flip)
            outputs.append(out)
        
        # 多尺度平均融合
        final_out = torch.stack(outputs).mean(dim=0)
        
        return final_out
    
    @torch.no_grad()
    def evaluate(self, test_loader):
        """
        评估整个测试集
        """
        correct, total = 0, 0
        to_pil = T.ToPILImage()
        
        for batch in tqdm(test_loader, desc="Dense Evaluating"):
            # 处理 batch 格式
            if isinstance(batch[0], list):
                images, labels = batch
            else:
                images = [to_pil(img) for img in batch[0]] if isinstance(batch[0], torch.Tensor) else batch[0]
                labels = batch[1]
            
            for img, label in zip(images, labels):
                # 确保是 PIL Image
                if isinstance(img, torch.Tensor):
                    img = to_pil(img)
                
                # 多尺度密集预测
                logits = self.predict_multiscale(img, flip=True)
                pred = logits.argmax().item()
                
                gt = label.item() if isinstance(label, torch.Tensor) else label
                correct += int(pred == gt)
                total += 1
        
        accuracy = correct / max(total, 1)
        print(f"\n✅ Dense Evaluation Accuracy: {accuracy:.4f} ({correct}/{total})")
        return accuracy

In [ ]:
evaluator = VGG19DenseEvaluator(dense_model, scales=[150, 275, 300], num_classes=101, device=device)